# Config Builder

Say you're just getting started with `pelagos`, or perhaps you're using someone else's data. You don't know what kind of procedures you should do, or what is even possible for the given file. Well, you're in luck. `pelagos-py` comes with a built in config builder in `config_builder.py` which can build a configuration for you based on the contents of the glider data file itself.

The default pipeline config (`default_config.yaml`) is a template that does everything: CTD, backscatter, oxygen, PAR, chlorophyll, and so on. A given glider file usually can't support all of what `pelagos-py` is built for - the glider might have no PAR sensor, ship raw beta values under the name `BBP700`, or carry optode `phase` that oxygen units could be recomputed from.

`pelagos_py.utils.config_builder` adapts the template to one file in three stages:

1. **Probe**: `file_probe.probe_file` reads which variables the file holds and their units.
2. **Decide**: `config_builder.decisions` turns the probe into a list of things the template must change. There are many types of decisions that can be made - so there are multiple helper functions to handle things like conductivity, backscatter, and oxygen.
3. **Build**: `config_builder.build` applies the choices to the template *text*, so the comments survive and the result is still a readable config.

This is all tied together in `pipeline.make_config` wraps all three. This notebook walks through each stage using two near realtime files, **Nelson 646** and **Doombar 648**, using the `get_demo_file` function.

In [1]:
import os
import pandas as pd

cwd = os.getcwd()
os.chdir(f"{cwd}/../../src")

from pelagos_py import Pipeline, get_demo_file          #   Pipeline and demo data access
from pelagos_py.utils import config_builder, file_probe #   What we're showing off here

14:24:02  discovery             Registered 27 steps and 10 QC tests successfully


## Get the demo files

`get_demo_file` downloads a specified demo file into `/examples/data/OG1/` the first time it's called and returns the path. Calling it with no name lists the available demos that are present in `demo_data.py`.

In [2]:
get_demo_file()

Bio-Carbon: nelson_646_r, nelson_646, doombar_648_r, doombar_648, churchill_647_r, churchill_647, alr_4_649_r, alr_4_649, alr_6_650_r, alr_6_650, cabot_645_r, cabot_645
Custard 1: churchill_501_r, churchill_501, pancake_502_r, doombar_503_r
Custard 2: bellamite_538_r, bellamite_538, zephyr_539
ReBELS: zephyr_675_r, zephyr_675, omg-1_676_r, 9ja_699_r, 9ja_699, growler_677_r, growler_677, stella_678_r, stella_678
ReBELS 2: stella_713_r
VOTO: sea063_20240724t0737_delayed


In [3]:
files = {
    "Nelson": get_demo_file("nelson_646_r"),
    "Doombar": get_demo_file("doombar_648_r"),
}
files

{'Nelson': PosixPath('/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Nelson_646_R.nc'),
 'Doombar': PosixPath('/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Doombar_648_R.nc')}

So we've got our data files, and we know the paths where we can find them.

## 1. Probe the files

The first thing that needs to be done in building the configuration automatically is to assess what is present in each file - that's the variables and associated units in the attributes.

In [4]:
probes = {name: file_probe.probe_file(path) for name, path in files.items()}
probes.keys()

dict_keys(['Nelson', 'Doombar'])

In [5]:
probe_table = pd.concat(
    {name: pd.DataFrame(probe).T for name, probe in probes.items()}, axis=1
)
probe_table.head(10)

Nelson                            \
                                       units numeric all_nan    median   
TIME      seconds since 1970-01-01T00:00:00Z    True   False       NaN   
TIME_GPS  seconds since 1970-01-01T00:00:00Z    True   False       NaN   
PHASE                                           True    True       NaN   
PHASE_QC                                        True    True       NaN   
CNDC                                  mhos/m    True   False    3.7177   
PRES                                 decibar    True   False       NaN   
TEMP                          degree_Celsius    True   False       NaN   
CHLA                                   mg/m3    True   False       NaN   
BBP700                                   m-1    True   False  0.000091   
BBP532                                   m-1    True   False       NaN   

                                     Doombar                            
                                       units numeric all_nan    median  
TIME      seconds since 1970-01-01T00:00:00Z    True   False       NaN  
TIME_GPS  seconds since 1970-01-01T00:00:00Z    True   False       NaN  
PHASE                                           True    True       NaN  
PHASE_QC                                        True    True       NaN  
CNDC                                  mhos/m    True   False   3.70574  
PRES                                 decibar    True   False       NaN  
TEMP                          degree_Celsius    True   False       NaN  
CHLA                                   mg/m3    True   False       NaN  
BBP700                                   m-1    True   False  0.000028  
BBP532                                   NaN     NaN     NaN       NaN

The two gliders don't carry the same sensors. Note how the "BBP532" line for Doombar is all NaN

Building the differences between the contents is easy to do using the file probes.

In [6]:
usable = {
    name: {v for v in probe if file_probe.present(probe, v)} for name, probe in probes.items()
}

print("Only in Nelson: ", sorted(usable["Nelson"] - usable["Doombar"]))
print("Only in Doombar:", sorted(usable["Doombar"] - usable["Nelson"]))

Only in Nelson:  ['BBP532', 'BBP532_QC', 'DOWNWELLING_PAR', 'DOWNWELLING_PAR_QC', 'RAW_DOWNWELLING_PAR', 'RAW_DOWNWELLING_PAR_QC']
Only in Doombar: ['CDOM', 'CDOM_QC']


So we can see that Nelson has some unique backscatter params (both have BBP700) and PAR. Whereas Doombar is equipped with CDOM. There is a PAR subroutine in `pelagos-py` (as of v0.0.0, October 2026), so given that Doombar doesn't have one, we'll see that a decision is made *not* to include the PAR processing in the final config file.

This means that we'll want to tweak our configuration files differently to accomodate for these different sensor payloads.

## 2. Decisions

`decisions(probe)` checks the file against what the template expects and returns one entry per required change:

| Decision example | Raised when |
|------------------|-------------|
| `coord_latitude` / `coord_longitude` | `_coordinate_decisions()`: `LATITUDE`/`LONGITUDE` is missing or mislabeled |
| `cndc` | `_cndc_decision()`: `CNDC` units don't match its values |
| `bbp` | `_bbp_decision`: looking for the `BETA_NAME` or `BBP_NAME` and providing instructions on how to handle them differently based on what is present |
| `oxygen` | `_oxygen_decision`: whether oxygen is recomputed from optode phase, used as shipped, or skipped |
| `par` | `DOWNWELLING_PAR` is missing |


So what do these decisions look like in practice?

In [7]:
config_builder.decisions(probes["Nelson"])

[{'id': 'bbp',
  'title': 'BETA_BACKSCATTERING700 missing, BBP700 present',
  'detail': "Some files ship raw beta under BBP700 before it has been converted. Either treat BBP700 as beta (renamed to BETA_BACKSCATTERING700 and converted by 'BBP from Beta') or trust it as already-converted BBP and skip the conversion.",
  'section': 'BACKSCATTER',
  'options': [{'key': 'as_beta',
    'label': 'Use BBP700 as raw beta and convert it'},
   {'key': 'direct', 'label': 'Use BBP700 directly, skip conversion'}],
  'default': 'as_beta'},
 {'id': 'oxygen',
  'title': 'Oxygen from DPHASE_DOXY',
  'detail': 'Full optode chain: phase -> pressure correction -> shift -> concentration. BPHASE_DOXY present but all-NaN.',
  'section': 'OXYGEN',
  'options': [{'key': 'phase', 'label': 'Recompute oxygen from DPHASE_DOXY'},
   {'key': 'shipped', 'label': 'Use MOLAR_DOXY as shipped'},
   {'key': 'none', 'label': 'No oxygen processing'}],
  'default': 'phase'}]

These decisions are able to be built directly from the probe and are formatted by some handy instructions:
* `sensor` - assigning an ID for indexing purposes (the 'type' is stored in `section`)
* `detail` - describing, in detail, what has been found from the probe and getting into what should be done with it
* `options` - often related to renaming or more specific items related to the detail

In [8]:
def show_decisions(decs):
    for d in decs:
        print(f"[{d['id']}] {d['title']}")
        print(f"    {d['detail']}")
        for o in d["options"]:
            # Rename options list every file variable; just show how many there are.
            if o["key"].startswith("rename:"):
                continue
            marker = "*" if o["key"] == d["default"] else " "   #   Highlight the default for later
            print(f"    {marker} {o['key']:<10} {o['label']}")
        n_renames = sum(o["key"].startswith("rename:") for o in d["options"])
        if n_renames:
            print(f"      + {n_renames} 'rename:<variable>' options")
        print()

In [9]:
all_decisions = {name: config_builder.decisions(probe) for name, probe in probes.items()}
for name, decs in all_decisions.items():
    print(f"===== {name} =====")
    show_decisions(decs)

===== Nelson =====
[bbp] BETA_BACKSCATTERING700 missing, BBP700 present
    Some files ship raw beta under BBP700 before it has been converted. Either treat BBP700 as beta (renamed to BETA_BACKSCATTERING700 and converted by 'BBP from Beta') or trust it as already-converted BBP and skip the conversion.
    * as_beta    Use BBP700 as raw beta and convert it
      direct     Use BBP700 directly, skip conversion

[oxygen] Oxygen from DPHASE_DOXY
    Full optode chain: phase -> pressure correction -> shift -> concentration. BPHASE_DOXY present but all-NaN.
    * phase      Recompute oxygen from DPHASE_DOXY
      shipped    Use MOLAR_DOXY as shipped
      none       No oxygen processing

===== Doombar =====
[bbp] BETA_BACKSCATTERING700 missing, BBP700 present
    Some files ship raw beta under BBP700 before it has been converted. Either treat BBP700 as beta (renamed to BETA_BACKSCATTERING700 and converted by 'BBP from Beta') or trust it as already-converted BBP and skip the conversion.
    *

Both files ship raw beta under `BBP700` and carry optode phase (`DPHASE_DOXY`), so they share the `bbp` and `oxygen` decisions. Doombar has no `DOWNWELLING_PAR`, so it gets an extra `par` decision whose default removes the PAR QC section.

The asterisk by each option indicates the **default** decision that can be made for each of these variables. Meanwhile, `config_builder.default_choices` collects the defaults into the `choices` dictionary that `build` takes:

In [10]:
{name: config_builder.default_choices(decs) for name, decs in all_decisions.items()}

{'Nelson': {'bbp': 'as_beta', 'oxygen': 'phase'},
 'Doombar': {'bbp': 'as_beta', 'oxygen': 'phase', 'par': 'remove'}}

Note how the asterisk items have been extracted. The only difference in this case is that Doombar's config shouldn't involve the PAR routine at all, given it wasn't detected in the probe.

## 3. Build the configs

`build(template_text, file_path, probe, choices)` returns the adapted config as a YAML string. Besides applying the choices, it fills in the input `file_path`, the export `output_path` (`<stem>_Processed.nc` next to the input) and the description.

With `choices=None` the defaults are used.

In [11]:
template = config_builder.DEFAULT_CONFIG.read_text()

configs = {
    name: config_builder.build(template, str(files[name]), probes[name])
    for name in files
}
print(configs["Nelson"][655:1600])  #   The copyright is automatically added

# Pipeline Configuration
pipeline:
  name: Complete processing pipeline
  description: "Pipeline built for Nelson_646_R.nc."
  out_directory: "/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/"
  log_file: complete_processing.log

# Steps in the pipeline
# For further details on step parameters, see all_step_configs.yaml
steps:

# ===========================================================
#                       IMPORT DATA
# ===========================================================
  - name: Load OG1
    parameters:
      file_path: "/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Nelson_646_R.nc"  # Path to the input NetCDF file
    diagnostics: false

# Normalise the file onto the names/units the steps below expect: LATITUDE from
# LATITUDE_GPS/ALATPT01 when missing, CNDC rescaled if mislabelled, and a raw
# beta shipped as BBP700 renamed to BETA_BACKSCATTERING700. Only what the file
# needs is applie


As you can see above, we've got our configuration, now with the `description` and `out_directory` updated for the general pipeline settings. The same goes for the Load OG1 step.

In [12]:
print(configs["Doombar"][655:1290])

# Pipeline Configuration
pipeline:
  name: Complete processing pipeline
  description: "Pipeline built for Doombar_648_R.nc."
  out_directory: "/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/"
  log_file: complete_processing.log

# Steps in the pipeline
# For further details on step parameters, see all_step_configs.yaml
steps:

# ===========================================================
#                       IMPORT DATA
# ===========================================================
  - name: Load OG1
    parameters:
      file_path: "/home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Doombar_648_R.nc"


The comments from the template yaml file are kept. To compare the two configs, let's list each one's steps, grouped by the template's section banners (`# ==== / # TITLE / # ====`).

`config_builder._parse` is the internal parser `build` uses to split the template into one block per step.

In [13]:
def steps_by_section(text):
    # head, blocks, tail
    _, blocks, _ = config_builder._parse(text)
    return pd.DataFrame([{"section": b.section, "step": b.name} for b in blocks])

In [14]:
#   What in the template is present in each of the glider datasets we've gotten?
template_steps = steps_by_section(template) #   full from teh tempate in default_config.yaml
step_table = template_steps.copy()
for name, text in configs.items():
    kept = steps_by_section(text).groupby(["section", "step"], dropna=False).size()
    step_table[name] = [
        "+" if (s, n) in kept.index else "-" for s, n in zip(step_table.section, step_table.step)
    ]
pd.set_option("display.max_rows", 200)
step_table

,section,step,Nelson,Doombar
0,IMPORT DATA,Load OG1,+,+
1,IMPORT DATA,Prepare OG1,+,+
2,IMPORT DATA,Format Checker,+,+
3,LATITUDE / LONGITUDE,Apply QC,+,+
4,CTD,Apply QC,+,+
5,TEMPERATURE CROSS CALIBRATION,Correct Values,+,+
6,INTERPOLATION,Interpolate Data,+,+
7,PROFILE FINDING,Derive CTD,+,+
8,PROFILE FINDING,Find Profiles,+,+
9,PROFILE FINDING,Apply QC,+,+


Only the steps that differ between the template and the two built configs:

In [15]:
step_table[(step_table["Nelson"] == "-") | (step_table["Doombar"] == "-")]

,section,step,Nelson,Doombar
27,PAR QC,Apply QC,+,-
28,PAR QC,Interpolate PAR,+,-


Which matches our results from the end of section 2.

## 4. Overriding the defaults in the build

As seen before, it's in `build` where we start to actually see our familiar YAML format showing up. However, we don't need to use the defaults for everything.

The `choices` argument in `build` allows us to pick a different option than the default. Anything not given keeps its default.

For example, to trust Nelson's `BBP700` variable as already-converted backscatter and use the oxygen concentration as shipped instead of recomputing it from phase:

In [16]:
custom = config_builder.build(
    template, str(files["Nelson"]), probes["Nelson"],
    choices={"bbp": "direct", "oxygen": "shipped"},
)

In [17]:
custom_steps = steps_by_section(custom)
for section in ["BACKSCATTER", "OXYGEN"]:
    # list(template_steps.query("section == @section").step)   #   type(custom_steps)
    print(f"{section}")
    print("  default:", list(template_steps.query("section == @section").step))
    print("  custom: ", list(custom_steps.query("section == @section").step))

BACKSCATTER
  default: ['Apply QC', 'BBP from Beta', 'Isolate BBP Spikes']
  custom:  ['Apply QC', 'Isolate BBP Spikes']
OXYGEN
  default: ['Derive Uncalibrated Phase', 'Phase Pressure Correction', 'Apply QC', 'Shift Oxygen To CTD', 'Derive Calibrated Phase', 'Derive Oxygen Concentration', 'Molar DOXY Salinity Correction', 'Molar DOXY Pressure Correction', 'Correct Values']
  custom:  ['Correct Values']


`bbp: direct` drops the `BBP from Beta` conversion, sets `bbp700_is_beta: false` in `Prepare OG1`, and points the backscatter steps at `BBP700`. `oxygen: shipped` keeps only the `Correct Values` step, applied to `MOLAR_DOXY`.

In [18]:
def show_step(text, step_name):
    # head, blocks, tail
    _, blocks, _ = config_builder._parse(text)
    block = next(b for b in blocks if b.name == step_name)  #   The object - use .text()
    print(block.text())

show_step(custom, "Prepare OG1")


# Normalise the file onto the names/units the steps below expect: LATITUDE from
# LATITUDE_GPS/ALATPT01 when missing, CNDC rescaled if mislabelled, and a raw
# beta shipped as BBP700 renamed to BETA_BACKSCATTERING700. Only what the file
# needs is applied, and each fix is logged.
  - name: Prepare OG1
    parameters:
      bbp700_is_beta: false
    diagnostics: false


### Renaming a variable

Decisions about a missing variable (`par`, `bbp`, `oxygen` when no oxygen is found, and missing coordinates) also offer `rename:<variable>`.

The builder adds a `renames:` mapping to `Prepare OG1`, which renames it before any other step runs.

Doombar has no PAR, so here is what choosing one of its variables in place of `DOWNWELLING_PAR` writes into the config:

In [19]:
par_options = next(d for d in all_decisions["Doombar"] if d["id"] == "par")["options"]
print([o["key"] for o in par_options])

['remove', 'rename:ALTITUDE', 'rename:BBP700', 'rename:CDOM', 'rename:CHLA', 'rename:CNDC', 'rename:DEPLOYMENT_LATITUDE', 'rename:DEPLOYMENT_LONGITUDE', 'rename:DEPLOYMENT_TIME', 'rename:DPHASE_DOXY', 'rename:GLIDER_DEPTH', 'rename:GLIDER_PITCH', 'rename:GLIDER_ROLL', 'rename:LATITUDE', 'rename:LATITUDE_GPS', 'rename:LONGITUDE', 'rename:LONGITUDE_GPS', 'rename:MOLAR_DOXY', 'rename:PARAMETER', 'rename:PARAMETER_SENSOR', 'rename:PARAMETER_UNITS', 'rename:PLATFORM_MODEL', 'rename:PLATFORM_TYPE', 'rename:PRES', 'rename:TEMP', 'rename:TIME', 'rename:TIME_GPS', 'rename:TRAJECTORY', 'rename:WATERCURRENTS_U', 'rename:WATERCURRENTS_V', 'rename:WMO_IDENTIFIER']


In [20]:
renamed = config_builder.build(
    template, str(files["Doombar"]), probes["Doombar"],
    choices={"par": "rename:CHLA"},  # illustration only: CHLA is not PAR
)
show_step(renamed, "Prepare OG1")


# Normalise the file onto the names/units the steps below expect: LATITUDE from
# LATITUDE_GPS/ALATPT01 when missing, CNDC rescaled if mislabelled, and a raw
# beta shipped as BBP700 renamed to BETA_BACKSCATTERING700. Only what the file
# needs is applied, and each fix is logged.
  - name: Prepare OG1
    parameters:
      bbp700_is_beta: true # Treat BBP700 with no BETA_BACKSCATTERING700 alongside as raw beta
      renames:  # file's name for a missing OG1 variable
        DOWNWELLING_PAR: CHLA
    diagnostics: false


In reality, you'd have something like `PAR_WAVELENGTH_OTHER_ABSTRACT_NAME` that can map onto what's missing.

## 5. All in one: `Pipeline.make_config`

In practice you'd use `Pipeline.make_config`, which probes the file, builds the config, saves it (by default as `<file name>.yaml` next to the file) and returns a `Pipeline` built from it. The defaults it chose are logged as warnings for extra visibility.

In [21]:
pipelines = {name: Pipeline.make_config(path) for name, path in files.items()}

14:24:04  pipeline              Logging to file: /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/complete_processing.log
14:24:04  pipeline              Pipeline initialised      [00:00<?]
14:24:04  pipeline              Replaced config /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Nelson_646_R.yaml
14:24:04  pipeline              BETA_BACKSCATTERING700 missing, BBP700 present: Use BBP700 as raw beta and convert it
14:24:04  pipeline              Oxygen from DPHASE_DOXY: Recompute oxygen from DPHASE_DOXY
14:24:04  pipeline              Logging to file: /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/complete_processing.log
14:24:04  pipeline              Pipeline initialised      [00:00<?]
14:24:04  pipeline              Replaced config /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Doombar_648_R.yaml
14:24:04  pipeline              BETA_BACKSCATTERING700 missing, BBP700 present: Use BBP700 as raw beta and convert it
14:24:04  pipe

In [22]:
for name, pipe in pipelines.items():
    print(f"{name}: {len(pipe.steps)} steps, config saved to {files[name].with_suffix('.yaml')}")

Nelson: 38 steps, config saved to /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Nelson_646_R.yaml
Doombar: 36 steps, config saved to /home/aaron-mau/Code/pelagos-py/pelagos-py/examples/data/OG1/Doombar_648_R.yaml


The saved config can be edited and rerun with `pipeline.load_config`. To process the files, run the pipelines as you normally would. In this case, this would suffice:

```python
for pipe in pipelines.values():
    pipe.run()
```

Or rerun individually by pointing to the new config files and calling `pipe.run()` once the pipeline is initialized.